# The Promise Gap: walkthrough

This notebook walks through the analysis end to end, reading the DuckDB warehouse that
`python run_all.py` builds. Every number here is computed live from the data.

Question: do late deliveries cause bad reviews and lost repeat purchases on Olist, and what should Olist test?

In [1]:
import json, os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import duckdb, pandas as pd, numpy as np
import plotly.graph_objects as go
from src import config, rd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
con = duckdb.connect(str(config.DB_PATH), read_only=True)
m = json.loads((config.OUTPUTS / 'metrics.json').read_text())
q = lambda sql: con.execute(sql).df()

## 1. Data profile

The key limitations: data ends in August 2018, 2016 is sparse, `customer_id` is per order,
and most late orders are reviewed before they arrive. Full detail in `docs/data_profile.md`.

In [2]:
pd.read_csv(config.OUTPUTS / 'profile' / 'tables.csv')

,table,rows,primary_key,distinct_keys,duplicate_key_rows,exact_duplicate_rows
0,orders,99441,order_id,99441,0,0
1,order_items,112650,"order_id, order_item_id",112650,0,0
2,customers,99441,customer_id,99441,0,0
3,order_reviews,99224,"review_id, order_id",99224,0,0
4,order_payments,103886,"order_id, payment_sequential",103886,0,0
5,products,32951,product_id,32951,0,0
6,sellers,3095,seller_id,3095,0,0
7,category_translation,71,category_pt,71,0,0
8,geolocation,1000163,"zip_prefix, lat, lng",720154,280009,261831
9,mql,8000,mql_id,8000,0,0


In [3]:
# Why customer_unique_id: customer_id has one value per order.
q('''select count(distinct customer_id) as customer_ids,
            count(distinct customer_unique_id) as people
     from stg_customers''')

,customer_ids,people
0,99441,96096


## 2. The clean model and exclusions

Grain: one row per delivered order. Every step that drops rows is logged.

In [4]:
q('select * from int_exclusions order by step')

,step,sample,description,rows_before,rows_after,rows_removed
0,1,Grain,All orders in the raw data,<NA>,99441,<NA>
1,2,Grain,Remove orders whose status is not delivered,99441,96478,2963
2,3,Grain,Remove delivered orders with no delivery times...,96478,96470,8
3,4,Reviews,Review outcomes: remove delivered orders with ...,96470,95824,646
4,5,Repeat,Repeat outcome: keep only each customer's firs...,96470,93350,3120
5,6,Repeat,Repeat outcome: remove first orders without a ...,93350,56034,37316
6,7,Cohorts,Customers (customer_unique_id) with at least o...,<NA>,93350,<NA>
7,8,Cohorts,Remove customers whose first purchase is befor...,93350,93086,264


## 3. Funnels

Stages are cumulative: an order that reached a later stage counts as having reached the earlier ones.

In [5]:
print((config.SQL_DIR / 'marts' / '02_mart_fulfilment_funnel.sql').read_text()[:1500])

-- Order fulfilment funnel: purchased > approved > shipped > delivered > reviewed.
-- One row per (dimension, value). GROUPING SETS produces the overall row,
-- the by-month rows and the by-category rows in a single pass.
create or replace table mart_fulfilment_funnel as
with stages as (
    -- Stages are cumulative: reaching a later stage implies the earlier ones.
    -- 14 delivered orders have no approval time but were clearly approved.
    select
        order_id,
        purchase_month,
        category_en,
        purchased_at,
        approved_at,
        shipped_at,
        delivered_at,
        review_answered_at,
        is_delivered                                           as reached_delivered,
        is_delivered or shipped_at is not null                 as reached_shipped,
        is_delivered or shipped_at is not null
            or approved_at is not null                         as reached_approved,
        is_delivered and review_id is not null                 as reac

In [6]:
q('''select dimension_value, n_purchased, n_approved, n_shipped, n_delivered, n_reviewed,
            round(conv_purchased_to_delivered, 4) as delivered_rate,
            round(median_h_ship_to_deliver, 1) as median_h_ship_to_deliver
     from mart_fulfilment_funnel where dimension = 'overall' ''')

,dimension_value,n_purchased,n_approved,n_shipped,n_delivered,n_reviewed,delivered_rate,median_h_ship_to_deliver
0,all,99441,99295,97659,96470,95824,0.9701,170.4


In [7]:
q('''select dimension_value, n_leads, n_closed, n_first_sale, n_first_sale_observed, n_still_selling,
            round(conv_lead_to_closed, 3) lead_to_closed, round(conv_closed_to_first_sale, 3) closed_to_sale,
            round(conv_first_sale_to_still_selling, 3) still_selling
     from mart_seller_funnel where dimension in ('overall', 'lead_origin') order by n_closed desc''')

,dimension_value,n_leads,n_closed,n_first_sale,n_first_sale_observed,n_still_selling,lead_to_closed,closed_to_sale,still_selling
0,all,8000,842,376,197,122,0.105,0.447,0.619
1,organic_search,2296,271,111,62,33,0.118,0.410,0.532
2,paid_search,1586,195,100,50,37,0.123,0.513,0.740
3,unknown,1159,193,84,45,28,0.167,0.435,0.622
4,social,1350,75,31,12,8,0.056,0.413,0.667
5,direct_traffic,499,56,31,16,9,0.112,0.554,0.563
6,referral,284,24,9,5,4,0.085,0.375,0.800
7,email,493,15,6,4,0,0.030,0.400,0.000
8,display,118,6,2,1,1,0.051,0.333,1.000
9,other,150,4,2,2,2,0.027,0.500,1.000


## 4. Cohort retention

Unobserved cells are NULL, not zero. The curve pools only cohorts that have reached each month.

In [8]:
tri = pd.read_csv(config.OUTPUTS / 'cohort_customers_triangle.csv', index_col=0)
(tri.iloc[:, :8] * 100).round(2)

,0,1,2,3,4,5,6,7
cohort_month,,,,,,,,
2017-01-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14
2017-02-01,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18
2017-03-01,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32
2017-04-01,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31
2017-05-01,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14
2017-06-01,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23
2017-07-01,100.0,0.53,0.35,0.24,0.29,0.21,0.32,0.11
2017-08-01,100.0,0.69,0.35,0.27,0.35,0.52,0.30,0.27
2017-09-01,100.0,0.70,0.55,0.27,0.45,0.22,0.22,0.25


In [9]:
print(f"Repeat rate, all customers, any time: {m['repeat_rate_all_time']:.2%}")
print(f"Repeat within 180 days (full window observed): {m['repeat_rate_180d']:.2%} of {m['repeat_rate_180d_base']:,} customers")
pd.read_csv(config.OUTPUTS / 'retention_curve_customers.csv').head(13)

Repeat rate, all customers, any time: 2.16%
Repeat within 180 days (full window observed): 1.98% of 56,034 customers


,month_number,active,base,n_cohorts,rate,cumulative_repeat_rate
0,0,86942,86942,19,1.000000,0.003738
1,1,389,80996,18,0.004803,0.008433
2,2,257,75121,17,0.003421,0.011741
3,3,180,68615,16,0.002623,0.014049
4,4,167,62033,15,0.002692,0.016378
5,5,132,55259,14,0.002389,0.018857
6,6,115,48971,13,0.002348,0.021074
7,7,86,42129,12,0.002041,0.023214
8,8,76,36791,11,0.002066,0.025876
9,9,55,29732,10,0.001850,0.028454


## 5. The promise gap

First the descriptive picture: outcome by days late.

In [10]:
df = q('select * from mart_promise_gap')
curve = df[df.days_late.between(-10, 15)].groupby('days_late').agg(
    orders=('order_id', 'size'), one_star=('one_star', 'mean'),
    review_before_delivery=('review_before_delivery', 'mean'), repeat_180=('repeat_180', 'mean'))
curve.round(3)

,orders,one_star,review_before_delivery,repeat_180
days_late,,,,
-10,4646,0.057,0.003,0.02
-9,4626,0.066,0.002,0.026
-8,4828,0.071,0.004,0.021
-7,4837,0.067,0.002,0.021
-6,3625,0.06,0.003,0.017
-5,2219,0.077,0.003,0.021
-4,1917,0.073,0.002,0.019
-3,1731,0.08,0.006,0.019
-2,1550,0.083,0.003,0.011


In [11]:
fig = go.Figure()
c = curve.reset_index()
fig.add_trace(go.Scatter(x=c.days_late, y=c.one_star, mode='markers+lines', name='1-star rate'))
fig.add_trace(go.Scatter(x=c.days_late, y=c.review_before_delivery, mode='markers+lines', name='Surveyed before delivery'))
fig.add_vline(x=0.5, line_dash='dash')
fig.update_layout(title='1-star rate rises with each day late; surveys start arriving before the parcel at day 3',
                  xaxis_title='Days late', yaxis_tickformat='.0%', height=420)
fig

Now the regression discontinuity. Bandwidth chosen by leave-one-out cross-validation,
then the local linear jump with HC1 standard errors. This re-runs the estimator live.

In [12]:
h, cv = rd.cv_bandwidth(df.days_late.values, df.one_star.values)
print('Chosen bandwidth:', h)
rows = []
for name in ['one_star', 'review_score', 'repeat_180']:
    r = rd.local_linear_rd(df.days_late.values, df[name].values, h=h)
    a = rd.adjacent_days(df.days_late.values, df[name].values)
    rows.append({'outcome': name, 'rd_jump': r['estimate'], 'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
                 'p': r['p_value'], 'day1_vs_day0': a['estimate']})
pd.DataFrame(rows).round(4)

Chosen bandwidth: 4


,outcome,rd_jump,ci_low,ci_high,p,day1_vs_day0
0,one_star,-0.0168,-0.0490,0.0154,0.3067,0.0453
1,review_score,-0.0899,-0.2240,0.0441,0.1885,-0.3019
2,repeat_180,0.0011,-0.0197,0.0219,0.9157,0.0033


In [13]:
# Validity checks
print('Density:', {k: round(v, 3) for k, v in rd.density_test(df.days_late.values, h=h).items()})
pd.read_csv(config.OUTPUTS / 'rd_balance.csv')[['covariate', 'estimate', 'p_value']].round(4)

Density: {'log_jump': np.float64(-0.384), 'se': np.float64(0.105), 'count_ratio': np.float64(0.681), 'ratio_ci_low': np.float64(0.555), 'ratio_ci_high': np.float64(0.837), 'p_value': np.float64(0.0), 'h': 4, 'count_day_0': 1292, 'count_day_1': 825}


,covariate,estimate,p_value
0,log_items_value,0.0415,0.3711
1,log_freight,0.0731,0.0101
2,n_items,0.0255,0.3214
3,promised_days,2.0130,0.0000
4,state_SP,-0.0574,0.0195
5,state_RJ,0.0157,0.3741
6,state_MG,0.0153,0.3128
7,state_RS,-0.0030,0.7672
8,state_PR,0.0006,0.9517
9,cat_bed_bath_table,0.0145,0.3422


Reading: no sharp jump at the promise date once the trend is modelled, but a steep ramp over
the next three days. Density and balance checks partly fail, so the local estimate carries a caveat.
This is a local effect at the promise date, not the effect of being a week late.

## 6. Business sizing

In [14]:
pd.read_csv(config.OUTPUTS / 'sizing.csv').round(2)

,scenario,scope,effect_pp,late_first_orders,repeat_customers_lost,repeat_value_per_repeater,loss_per_quarter
0,low,local,2.19,313,-6.87,162.37,-1114.72
1,low,all_late,2.19,799,-17.53,162.37,-2845.56
2,base,local,0.11,313,-0.35,162.37,-57.12
3,base,all_late,0.11,799,-0.90,162.37,-145.82
4,high,local,-1.97,313,6.16,162.37,1000.47
5,high,all_late,-1.97,799,15.73,162.37,2553.92
6,naive_reference,all_late,-0.41,799,3.27,162.37,531.66


## 7. Experiment design (not run)

In [15]:
pd.read_csv(config.OUTPUTS / 'experiment_power.csv').round(2)

,relative_drop_in_one_star,absolute_drop_pp,n_per_arm,total,months_at_measured_volume
0,0.05,2.8,4960,9920,22.55
1,0.10,5.6,1245,2490,5.66
2,0.15,8.4,554,1108,2.52
3,0.20,11.2,312,624,1.42
4,0.30,16.8,138,276,0.63


In [16]:
print(f"Primary metric baseline (1-star among late orders): {m['exp_p_one_star_late']:.1%}")
print(f"Required per arm: {m['exp_n_per_arm']:,}  total: {m['exp_n_total']:,}  months: {m['exp_months']:.1f}")

Primary metric baseline (1-star among late orders): 56.0%
Required per arm: 1,245  total: 2,490  months: 5.7


In [17]:
con.close()